# Lab 2: Slider-Controlled Threat Rings

This lab builds on the click-handler pattern from Lab 1.
Now students will:

- click to place a launcher
- create a threat ring centered on that click
- change the ring radius with a slider
- change the color with a dropdown
- update the map live without rebuilding it

## Learning Goals

- use `Circle` to represent a real-world distance in meters
- connect widgets to callbacks with `observe`
- update an existing layer instead of recreating the whole map

In [ ]:
from ipyleaflet import Map, Marker, Circle, LayersControl, WidgetControl, basemaps, basemap_to_tiles
import ipywidgets as widgets
from IPython.display import display

## Build the Controls

The slider controls radius in meters.
The dropdown controls ring color.

In [ ]:
radius_slider = widgets.IntSlider(
    value=25000,
    min=5000,
    max=100000,
    step=5000,
    description="Radius",
)

color_menu = widgets.Dropdown(
    options=["red", "orange", "blue", "green"],
    value="red",
    description="Color",
)

status = widgets.HTML(value="<b>Status:</b> Click the map to place a launcher.")
panel = widgets.VBox([radius_slider, color_menu, status])

In [ ]:
m = Map(center=(33.9137, -98.4934), zoom=6)
m.add(basemap_to_tiles(basemaps.OpenStreetMap.Mapnik))
m.add(LayersControl())
m.add(WidgetControl(widget=panel, position="topright"))

launcher_marker = None
threat_ring = None

display(m)

## Event Functions

There are two kinds of events here:

- map click events
- widget value changes

In [ ]:
def update_ring(change=None):
    if threat_ring is None:
        return

    threat_ring.radius = radius_slider.value
    threat_ring.color = color_menu.value
    threat_ring.fill_color = color_menu.value


def handle_map_click(**kwargs):
    global launcher_marker, threat_ring

    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]

    if launcher_marker is not None:
        m.remove(launcher_marker)
    if threat_ring is not None:
        m.remove(threat_ring)

    launcher_marker = Marker(location=(lat, lon))
    threat_ring = Circle(
        location=(lat, lon),
        radius=radius_slider.value,
        color=color_menu.value,
        fill_color=color_menu.value,
        fill_opacity=0.2,
    )

    m.add(launcher_marker)
    m.add(threat_ring)

    status.value = (
        f"<b>Status:</b> Launcher at ({lat:.4f}, {lon:.4f}) with "
        f"a {radius_slider.value:,} meter threat ring."
    )


radius_slider.observe(update_ring, names="value")
color_menu.observe(update_ring, names="value")
m.on_interaction(handle_map_click)

## Questions

1. What happens if the slider changes before the user clicks the map?
2. Why is `Circle` better than a plain `Marker` for a threat zone?
3. Why is it cleaner to update `threat_ring.radius` than to recreate the whole map?

## Mini Challenges

1. Add a checkbox that toggles the ring on and off.
2. Add a second ring to represent a warning zone.
3. Display the radius in kilometers as well as meters.